# DSPG (Sheehan, гл. 7 Čápek–Sheehan 2005): независимая перепроверка условия Вегшайдера

**Что делает ноутбук.**
1. Заново вычисляет условие Вегшайдера $W$ для параметров табл. 7.1–7.2 по ур. 7.15–7.21 (свой код, без numpy/scipy). Сверяет результат с прежней фоновой задачей (`tmp/capek/ch6_7_checks.py`, `tmp/capek/ch6_7.md`, $W\approx10^{26}$).
2. Воспроизводит стационарное решение и табл. 7.3, ΔP по ур. 7.25/7.26 и циклический ток.
3. Показывает, что при $W=1$ (поправленные предэкспоненты) $\Delta P\to0$ и ток $\to0$, а знак ΔP совпадает со знаком $\ln W$.
4. Отдельным разделом проверяет вычислимые утверждения из ноутбука пользователя `chemical_nonequilibrium_steady_states.ipynb`.

**Источник.** `tmp/capek/capek.pdf`. Таблицы 7.1–7.3 — это картинки; я прочитал их с отрендеренных страниц.
**Пагинация:** стр. книги = PDF − 14. Табл. 7.1–7.2 — стр. 222 книги (PDF 236), табл. 7.3 — стр. 224 (PDF 238), ур. 7.15–7.21 — стр. 220–221 (PDF 234–235). Предположение «PDF + 2» неверно.

**Рабочий критерий.** Ток при равных термодинамических силах возможен только при нарушении условия Вегшайдера по циклу ($W\ne1$), то есть детального равновесия или гиббсовости термостата.

Каждая проверка печатает `[OK]` или `[FAIL]`. Итог подводится в последней ячейке.

In [1]:
preparser(False)

In [2]:
from math import sqrt, exp, log, log10, pi
from fractions import Fraction

CHECKS = []
def check(name, cond, detail=''):
    st = 'OK' if cond else 'FAIL'
    CHECKS.append((name, st))
    print('[%s] %s%s' % (st, name, ('   (' + detail + ')') if detail else ''))
    return cond

def rel(a, b):
    return abs(a - b) / abs(b)

def within(a, b, tol):
    return rel(a, b) <= tol

# CODATA 2018 (точные значения СИ) + а.е.м.
kB = 1.380649e-23; NA = 6.02214076e23; h = 6.62607015e-34
Rg = kB * NA; amu = 1.66053906660e-27
Torr = 101325.0 / 760.0
check('R = kB*NA = 8.314462618 Дж/(моль К)', abs(Rg - 8.314462618) < 1e-8, '%.9f' % Rg)

[OK] R = kB*NA = 8.314462618 Дж/(моль К)   (8.314462618)


## 1. Исходные данные (прочитано с рендера стр. 222 книги)

**Табл. 7.1.**

| параметр | значение |
|---|---|
| $m_{A_2}$, $m_A$ | 40 а.е.м., 20 а.е.м. |
| $r_A$ | $5\cdot10^{-10}$ м |
| $v_{A_2}$, $v_A$ («RMS») | 790 м/с, $1.1\cdot10^3$ м/с |
| «Cavity A₂ density $n(2,A_2)$» | $2\cdot10^{16}$ м⁻³ (по смыслу $n(c,A_2)$; $n(2,A_2)$ — это поверхностная плотность из табл. 7.3) |
| $T$ | 1000 K |
| $R_{cav}$; масштаб пятна S1 | 0.1 м; $10^{-3}$ м |
| $(SA)_2/(SA)_1$ | $10^9$ |
| $E(A\!-\!A)$ | 240 кДж/моль |
| $\nu_0$; $\nu_{vib}$ | $10^{13}$ Гц; $10^{13}$ Гц |
| монослой | $10^{19}$ м⁻² |

**Табл. 7.2** (все ΔE в кДж/моль).

| | S1 | S2 |
|---|---|---|
| $\Delta E_{des}(A)$ | 250 | 200 |
| $\Delta E_{des}(A_2)$ | 260 | 190 |
| $\Delta E_{diss,act}$ | 0 | 30 |
| $F(A)$ | $10^{-2}$ | $10^{3}$ |
| $F(A_2)$ | $10^{3}$ | 1 |
| $\gamma_{diss}$ | $10^{-1}$ | $10^{-9}$ |
| $\gamma_{recomb}$ | $10^{-6}$ | $10^{-1}$ |

Ниже две проверки согласованности. Скорости соответствуют RMS при 1000 K. Энергии согласованы по закону Гесса: энергия реакции $A_2(s)\to2A(s)$, вычисленная через $E(A\!-\!A)$ и энергии десорбции, совпадает с $\Delta E_{diss,act}$.

In [3]:
T0 = 1000.0; nu0 = 1e13; nuvib = 1e13; rA = 5e-10
vA = 1.1e3; vA2 = 790.0; mA = 20 * amu; mA2 = 40 * amu
EAA = 240.0                      # кДж/моль
area_ratio = 1e9                 # (SA)2/(SA)1
n_mono = 1e19
ncA2_ref = 2e16                  # n(c,A2), м^-3
S = {1: dict(EdA=250., EdA2=260., Ediss=0.,  FA=1e-2, FA2=1e3, gd=1e-1, gr=1e-6),
     2: dict(EdA=200., EdA2=190., Ediss=30., FA=1e3,  FA2=1.0, gd=1e-9, gr=1e-1)}

v_rms = lambda m, T=T0: sqrt(3 * kB * T / m)
check('v_rms(A2, 1000 K) = 790 м/с (табл. 7.1)', within(v_rms(mA2), 790., 0.01), '%.1f' % v_rms(mA2))
check('v_rms(A, 1000 K) = 1.1e3 м/с (табл. 7.1)', within(v_rms(mA), 1.1e3, 0.03), '%.1f' % v_rms(mA))
check('v_A/v_A2 = sqrt(2) с точностью 2%', within(vA / vA2, sqrt(2), 0.02), '%.4f' % (vA / vA2))
for i in (1, 2):
    p = S[i]
    dU = EAA - 2 * p['EdA'] + p['EdA2']          # A2(s) -> 2A(s), кДж/моль
    check('S%d: Гесс  E(A-A) - 2dE_des(A) + dE_des(A2) = dE_diss,act' % i,
          abs(dU - p['Ediss']) < 1e-12, '%g = %g кДж/моль' % (dU, p['Ediss']))

[OK] v_rms(A2, 1000 K) = 790 м/с (табл. 7.1)   (789.7)
[OK] v_rms(A, 1000 K) = 1.1e3 м/с (табл. 7.1)   (1116.8)
[OK] v_A/v_A2 = sqrt(2) с точностью 2%   (1.3924)
[OK] S1: Гесс  E(A-A) - 2dE_des(A) + dE_des(A2) = dE_diss,act   (0 = 0 кДж/моль)
[OK] S2: Гесс  E(A-A) - 2dE_des(A) + dE_des(A2) = dE_diss,act   (30 = 30 кДж/моль)


## 2. Константы и условие Вегшайдера

Уравнения книги (стр. 220–221):
- (7.20): $\tau_{des}(i,A_j)=\nu_0^{-1}F(i,A_j)\,e^{\Delta E_{des}(i,A_j)/kT}$ — **с** больцмановским множителем;
- (7.21): $K(i)=\dfrac{n(i,A_2)}{n^2(i,A)}=\dfrac{r_Av_A}{\nu_{vib}}\dfrac{\gamma_{recomb}(i)}{\gamma_{diss}(i)}e^{\Delta E_{diss,act}(i)/kT}$;
- (7.15), (7.16): $n(c,A_j)=\dfrac{4\,n(2,A_j)}{v_{A_j}\tau_{des}(2,A_j)}$, т.е. баланс «падает = десорбируется» при прилипании 1 (допущение b).

Из (7.15)/(7.16), записанных для поверхности $i$, следуют константы распределения газ↔поверхность $H_j(i)\equiv n(i,A_j)/n(c,A_j)=v_{A_j}\tau_{des}(i,A_j)/4$. Если газ находится в равновесии **только** с поверхностью $i$ (запаянная полость BBC(i) из §7.2), его константа равна
$$K_g(i)=\frac{n(c,A_2)}{n^2(c,A)}=K(i)\,\frac{H_A(i)^2}{H_{A_2}(i)}.$$

Рассмотрим цикл $A_2(g)\xrightarrow{S1}2A(g)\xrightarrow{S2}A_2(g)$. Произведение констант по циклу равно
$$W=\frac{K_g(2)}{K_g(1)}.$$
Детальное равновесие при одной $T$ (катализатор не сдвигает равновесие) требует $W=1$.

In [4]:
def tau_des(F, E, T=T0):                        # (7.20)
    return F / nu0 * exp(E * 1e3 / (Rg * T))

def K_surf(p, T=T0):                            # (7.21)
    return rA * vA / nuvib * p['gr'] / p['gd'] * exp(p['Ediss'] * 1e3 / (Rg * T))

def surface(p, T=T0):
    d = {}
    d['tA'] = tau_des(p['FA'], p['EdA'], T)
    d['tA2'] = tau_des(p['FA2'], p['EdA2'], T)
    d['K'] = K_surf(p, T)
    d['HA'] = vA * d['tA'] / 4                  # n(i,A)/n(c,A)
    d['HA2'] = vA2 * d['tA2'] / 4               # n(i,A2)/n(c,A2)
    d['Kg'] = d['K'] * d['HA']**2 / d['HA2']    # газ, равновесный с одной поверхностью i
    return d

sf = {i: surface(S[i]) for i in (1, 2)}
for i in (1, 2):
    d = sf[i]
    print('S%d: tau_A=%.4g c  tau_A2=%.4g c  K=%.4g м^2  H_A=%.4g м  H_A2=%.4g м  Kg=%.4g м^3'
          % (i, d['tA'], d['tA2'], d['K'], d['HA'], d['HA2'], d['Kg']))

# сверка с табл. 7.3 (стр. 224 книги)
tab73_tau = {(1, 'tA'): 0.012, (1, 'tA2'): 4000., (2, 'tA'): 2.9, (2, 'tA2'): 8.7e-4}
for (i, k), v in tab73_tau.items():
    check('табл. 7.3: %s(S%d) = %g c (допуск 10%%)' % (k, i, v), within(sf[i][k], v, 0.10), '%.4g' % sf[i][k])

W = sf[2]['Kg'] / sf[1]['Kg']
print('\nW = Kg(2)/Kg(1) = %.6g' % W)

S1: tau_A=0.01144 c  tau_A2=3808 c  K=5.5e-25 м^2  H_A=3.146 м  H_A2=7.522e+05 м  Kg=7.237e-30 м^3
S2: tau_A=2.797 c  tau_A2=0.0008402 c  K=2.029e-10 м^2  H_A=769.2 м  H_A2=0.1659 м  Kg=0.0007237 м^3
[OK] табл. 7.3: tA(S1) = 0.012 c (допуск 10%)   (0.01144)
[OK] табл. 7.3: tA2(S1) = 4000 c (допуск 10%)   (3808)
[OK] табл. 7.3: tA(S2) = 2.9 c (допуск 10%)   (2.797)
[OK] табл. 7.3: tA2(S2) = 0.00087 c (допуск 10%)   (0.0008402)

W = Kg(2)/Kg(1) = 1e+26


### 2а. Точное значение $W$

В $K_g(i)$ входит показатель $\exp\{[\Delta E_{diss,act}+2\Delta E_{des}(A)-\Delta E_{des}(A_2)]/RT\}$. По Гессу (п. 1) он равен $e^{E(A-A)/RT}$ для **обеих** поверхностей. Кроме того, $r_A$, $v$, $\nu_0$ и $\nu_{vib}$ общие. Поэтому
$$W=\frac{[(\gamma_r/\gamma_d)\,F_A^2/F_{A_2}]_{S2}}{[(\gamma_r/\gamma_d)\,F_A^2/F_{A_2}]_{S1}}$$
— это чистое число из табл. 7.2, **не зависящее от $T$**. Ниже оно вычислено в точной рациональной арифметике.

In [5]:
def pref_exact(i):
    q = {1: dict(gr=Fraction(1, 10**6), gd=Fraction(1, 10), FA=Fraction(1, 100), FA2=Fraction(1000)),
         2: dict(gr=Fraction(1, 10),    gd=Fraction(1, 10**9), FA=Fraction(1000), FA2=Fraction(1))}[i]
    return q['gr'] / q['gd'] * q['FA']**2 / q['FA2']

expo = {i: S[i]['Ediss'] + 2 * S[i]['EdA'] - S[i]['EdA2'] for i in (1, 2)}
check('показатели экспонент в Kg(1), Kg(2) равны (= E(A-A))', expo[1] == expo[2] == EAA, str(expo))
W_exact = pref_exact(2) / pref_exact(1)
print('предэкспоненты: S1 = %s, S2 = %s' % (pref_exact(1), pref_exact(2)))
check('W (точно) = 10^26', W_exact == 10**26, 'W_exact = %s' % W_exact)
check('W (float) совпадает с точным до 1e-9', rel(W, float(W_exact)) < 1e-9, '%.10g' % W)
WT = [surface(S[2], T)['Kg'] / surface(S[1], T)['Kg'] for T in (300., 1000., 3000.)]
check('W не зависит от T (300, 1000, 3000 K)', all(rel(w, 1e26) < 1e-9 for w in WT), ', '.join('%.6g' % w for w in WT))
check('условие Вегшайдера W = 1 НАРУШЕНО (|log10 W| > 1)', abs(log10(W)) > 1, 'log10 W = %.3f' % log10(W))

[OK] показатели экспонент в Kg(1), Kg(2) равны (= E(A-A))   ({1: 240.0, 2: 240.0})
предэкспоненты: S1 = 1/1000000000000, S2 = 100000000000000
[OK] W (точно) = 10^26   (W_exact = 100000000000000000000000000)
[OK] W (float) совпадает с точным до 1e-9   (1e+26)
[OK] W не зависит от T (300, 1000, 3000 K)   (1e+26, 1e+26, 1e+26)
[OK] условие Вегшайдера W = 1 НАРУШЕНО (|log10 W| > 1)   (log10 W = 26.000)


### 2б. Сверка с прежней фоновой задачей

Прежний вывод `tmp/capek/ch6_7_checks.py` (я перезапустил его в этой сессии, он совпадает с приложением к `ch6_7.md`):
`Kg(1)=7.25e-30, Kg(2)=7.25e-4 м³, W=1e+26; alpha1=1.5e-13, alpha2=2.73e+06, dP=8.45e-05 Па, J=3.95e+18 м⁻²с⁻¹; при W=1 dP=2.6e-27 Па`.
Мой код написан заново: другие константы (CODATA вместо R=8.314, amu=1.6605e-27), бисекция вместо формулы корней квадратного уравнения, точная рациональная арифметика для W. Числа сверяются с допуском 1% там, где они появляются.

In [6]:
prev = dict(Kg1=7.25e-30, Kg2=7.25e-4, W=1e26, a1=1.5e-13, a2=2.73e6, dP=8.45e-5, J=3.95e18)
check('Kg(1) совпадает с прежним (1%)', within(sf[1]['Kg'], prev['Kg1'], 0.01), '%.4g' % sf[1]['Kg'])
check('Kg(2) совпадает с прежним (1%)', within(sf[2]['Kg'], prev['Kg2'], 0.01), '%.4g' % sf[2]['Kg'])
check('W совпадает с прежним 1e26', within(W, prev['W'], 1e-6), '%.6g' % W)

[OK] Kg(1) совпадает с прежним (1%)   (7.237e-30)
[OK] Kg(2) совпадает с прежним (1%)   (0.0007237)
[OK] W совпадает с прежним 1e26   (1e+26)


## 3. Стационарное решение (7.15)–(7.19), табл. 7.3, ΔP (7.25)/(7.26), циклический ток

Площадь S2 в $10^9$ раз больше, поэтому газ задаёт S2:
$n(2,A_2)=H_{A_2}(2)\,n(c,A_2)$ (7.16), $n(2,A)=\sqrt{n(2,A_2)/K(2)}$ (7.18), $n(c,A)=n(2,A)/H_A(2)$ (7.15).
Полный поток атомов на поверхность: $R_T=\tfrac14[v_An(c,A)+2v_{A_2}n(c,A_2)]$ (стр. 223).
Для S1 из (7.17)+(7.19): $n(1,A)/\tau_A+2K(1)n(1,A)^2/\tau_{A_2}=R_T$. Корень нахожу бисекцией.

Обозначения: $\alpha(i)=R_{des}(i,A_2)/R_{des}(i,A)$ (7.22); циклический ток $J=\Gamma_{A_2}-R_{des}(1,A_2)$ — это нетто-поглощение $A_2$ на S1 (= нетто-рекомбинация на S2), где $\Gamma_{A_2}=v_{A_2}n(c,A_2)/4$.

In [7]:
def steady(ncA2, s1, s2):
    r = {}
    r['n2A2'] = s2['HA2'] * ncA2                     # (7.16)
    r['n2A'] = sqrt(r['n2A2'] / s2['K'])             # (7.18)
    r['ncA'] = r['n2A'] / s2['HA']                   # (7.15)
    r['ncA2'] = ncA2
    GA, GA2 = vA * r['ncA'] / 4, vA2 * ncA2 / 4
    RT = GA + 2 * GA2
    f = lambda x: x / s1['tA'] + 2 * s1['K'] * x * x / s1['tA2'] - RT
    lo, hi = 0.0, RT * s1['tA']                      # f(lo) < 0 <= f(hi)
    for _ in range(300):
        mid = 0.5 * (lo + hi)
        if f(mid) < 0: lo = mid
        else: hi = mid
    r['n1A'] = 0.5 * (lo + hi)
    r['n1A2'] = s1['K'] * r['n1A']**2                # (7.17)
    r['R1A'] = r['n1A'] / s1['tA'];  r['R1A2'] = r['n1A2'] / s1['tA2']
    r['R2A'] = r['n2A'] / s2['tA'];  r['R2A2'] = r['n2A2'] / s2['tA2']
    r['a1'] = r['R1A2'] / r['R1A'];  r['a2'] = r['R2A2'] / r['R2A']
    r['GA'], r['GA2'], r['RT'] = GA, GA2, RT
    r['J'] = GA2 - r['R1A2']                         # нетто-поглощение A2 на S1
    r['J_alt'] = (r['R1A'] - GA) / 2                 # то же через атомы
    r['dP25'] = mA*vA*r['R1A'] + mA2*vA2*r['R1A2'] - mA*vA*r['R2A'] - mA2*vA2*r['R2A2']   # (7.25)
    a1, a2 = r['a1'], r['a2']
    r['dP26'] = (2 - sqrt(2)) * mA * vA * RT * (a2 - a1) / ((2*a1 + 1) * (2*a2 + 1))     # (7.26)
    r['P'] = (ncA2 + r['ncA']) * kB * T0
    return r

st = steady(ncA2_ref, sf[1], sf[2])
for k in ('n1A', 'n1A2', 'n2A', 'n2A2', 'ncA', 'R1A', 'R1A2', 'R2A', 'R2A2', 'a1', 'a2', 'J', 'dP25', 'dP26', 'P'):
    print('%-5s = %.4g' % (k, st[k]))

n1A   = 9.037e+16
n1A2  = 4.492e+09
n2A   = 4.044e+12
n2A2  = 3.319e+15
ncA   = 5.257e+09
R1A   = 7.9e+18
R1A2  = 1.179e+06
R2A   = 1.446e+12
R2A2  = 3.95e+18
a1    = 1.493e-13
a2    = 2.732e+06
J     = 3.95e+18
dP25  = 8.133e-05
dP26  = 8.453e-05
P     = 0.0002761


In [8]:
tab73 = {'n1A': 8.8e16, 'n2A': 4e12, 'n1A2': 4.2e9, 'n2A2': 3.2e15,
         'R1A': 7.3e18, 'R2A': 1.4e12, 'R1A2': 1.1e6, 'R2A2': 3.7e18, 'a1': 1.4e-13, 'a2': 2.6e6}
for k, v in tab73.items():
    check('табл. 7.3: %s = %g (допуск 15%%)' % (k, v), within(st[k], v, 0.15), '%.3g' % st[k])
for k, v in (('n1A', 8.8e-3), ('n2A2', 3.2e-4)):
    check('табл. 7.3: theta(%s) = %g' % (k, v), within(st[k] / n_mono, v, 0.15), '%.3g' % (st[k] / n_mono))
check('стр. 227: R_des(2,A2)/R_des(1,A2) = 3.4e12', within(st['R2A2'] / st['R1A2'], 3.4e12, 0.15), '%.3g' % (st['R2A2'] / st['R1A2']))
check('стр. 227: R_des(2,A)/R_des(1,A) = 1.9e-7', within(st['R2A'] / st['R1A'], 1.9e-7, 0.15), '%.3g' % (st['R2A'] / st['R1A']))
check('(7.23), (7.24): оба отношения >> (SA)1/(SA)2 = 1e-9',
      st['R2A2'] / st['R1A2'] > 100 / area_ratio and st['R2A'] / st['R1A'] > 100 / area_ratio)

# tau_diss, tau_recomb из табл. 7.3: в тексте формул нет; реконструкция, согласованная с (7.21)
tdiss = {i: exp(S[i]['Ediss'] * 1e3 / (Rg * T0)) / (nuvib * S[i]['gd']) for i in (1, 2)}
trec = {i: 1 / ((st['n1A'] if i == 1 else st['n2A']) * 2 * rA * vA * S[i]['gr']) for i in (1, 2)}
check('реконструкция tau_diss = e^{E/kT}/(nu_vib*g_d) даёт табл. 7.3 (1e-12; 3.7e-3)',
      within(tdiss[1], 1e-12, 0.15) and within(tdiss[2], 3.7e-3, 0.15), '%.3g; %.3g' % (tdiss[1], tdiss[2]))
check('реконструкция tau_rec = 1/(n_A*2r_A*v_A*g_r) даёт табл. 7.3 (1e-5; 2.3e-6)',
      within(trec[1], 1e-5, 0.15) and within(trec[2], 2.3e-6, 0.15), '%.3g; %.3g' % (trec[1], trec[2]))
Kchk = {i: (rA * vA * S[i]['gr']) * exp(S[i]['Ediss'] * 1e3 / (Rg * T0)) / (nuvib * S[i]['gd']) for i in (1, 2)}
check('баланс n(A2)/tau_diss = n(A)/(2 tau_rec) воспроизводит K(i) по (7.21)',
      all(within(Kchk[i], sf[i]['K'], 1e-12) for i in (1, 2)))
check('Limit 4: tau_rec(1) << tau_des(1,A)', trec[1] < 1e-2 * sf[1]['tA'], '%.2g vs %.2g' % (trec[1], sf[1]['tA']))

[OK] табл. 7.3: n1A = 8.8e+16 (допуск 15%)   (9.04e+16)
[OK] табл. 7.3: n2A = 4e+12 (допуск 15%)   (4.04e+12)
[OK] табл. 7.3: n1A2 = 4.2e+09 (допуск 15%)   (4.49e+09)
[OK] табл. 7.3: n2A2 = 3.2e+15 (допуск 15%)   (3.32e+15)
[OK] табл. 7.3: R1A = 7.3e+18 (допуск 15%)   (7.9e+18)
[OK] табл. 7.3: R2A = 1.4e+12 (допуск 15%)   (1.45e+12)
[OK] табл. 7.3: R1A2 = 1.1e+06 (допуск 15%)   (1.18e+06)
[OK] табл. 7.3: R2A2 = 3.7e+18 (допуск 15%)   (3.95e+18)
[OK] табл. 7.3: a1 = 1.4e-13 (допуск 15%)   (1.49e-13)
[OK] табл. 7.3: a2 = 2.6e+06 (допуск 15%)   (2.73e+06)
[OK] табл. 7.3: theta(n1A) = 0.0088   (0.00904)
[OK] табл. 7.3: theta(n2A2) = 0.00032   (0.000332)
[OK] стр. 227: R_des(2,A2)/R_des(1,A2) = 3.4e12   (3.35e+12)
[OK] стр. 227: R_des(2,A)/R_des(1,A) = 1.9e-7   (1.83e-07)
[OK] (7.23), (7.24): оба отношения >> (SA)1/(SA)2 = 1e-9
[OK] реконструкция tau_diss = e^{E/kT}/(nu_vib*g_d) даёт табл. 7.3 (1e-12; 3.7e-3)   (1e-12; 0.00369)
[OK] реконструкция tau_rec = 1/(n_A*2r_A*v_A*g_r) даёт табл. 7.

In [9]:
check('баланс атомов на S1: два выражения J совпадают', rel(st['J'], st['J_alt']) < 1e-9, '%.6g / %.6g' % (st['J'], st['J_alt']))
# (7.26) выводится из (7.25) при m_A2 v_A2 = sqrt2 m_A v_A и R_des(i,A) + 2 R_des(i,A2) = R_T
dP25c = mA*vA*(st['R1A'] - st['R2A']) + sqrt(2)*mA*vA*(st['R1A2'] - st['R2A2'])
check('(7.26) = (7.25) точно при m_A2 v_A2 = sqrt2 m_A v_A', rel(st['dP26'], dP25c) < 1e-9, '%.6g vs %.6g Па' % (st['dP26'], dP25c))
check('баланс потоков: R_des(i,A) + 2 R_des(i,A2) = R_T на S1 и S2',
      rel(st['R1A'] + 2*st['R1A2'], st['RT']) < 1e-9 and rel(st['R2A'] + 2*st['R2A2'], st['RT']) < 1e-9)
print('  с табличными скоростями (m_A2 v_A2 / m_A v_A = %.4f вместо sqrt2): (7.25) = %.4g Па, т.е. на %.1f%% меньше (7.26)'
      % (mA2*vA2/(mA*vA), st['dP25'], 100*(1 - st['dP25']/st['dP26'])))
check('dP совпадает с прежним 8.45e-5 Па (1%)', within(st['dP26'], prev['dP'], 0.01), '%.4g' % st['dP26'])
check('alpha1, alpha2 совпадают с прежними (2%)', within(st['a1'], prev['a1'], 0.05) and within(st['a2'], prev['a2'], 0.02),
      '%.3g, %.3g' % (st['a1'], st['a2']))
check('циклический ток J совпадает с прежним 3.95e18 (1%)', within(st['J'], prev['J'], 0.01), '%.4g м^-2 с^-1' % st['J'])
q = st['J'] * EAA * 1e3 / NA
print('тепло, поглощаемое S1 (и выделяемое S2) при одной T: %.3g Вт/м^2; на пятно 1e-6 м^2: %.3g Вт' % (q, q * 1e-6))
print('dP/P = %.3g' % (st['dP26'] / st['P']))

# диапазон dP на стр. 225: 8e-7 ... 8e-4 Па при 2e14 ... 2e17 м^-3
lo_, hi_ = steady(2e14, sf[1], sf[2])['dP26'], steady(2e17, sf[1], sf[2])['dP26']
check('стр. 225: dP(2e14) ~ 8e-7 Па и dP(2e17) ~ 8e-4 Па (в пределах x1.5)',
      within(lo_, 8e-7, 0.5) and within(hi_, 8e-4, 0.5), '%.3g; %.3g Па' % (lo_, hi_))

# инвентари (стр. 226–227) — неявно задают S2 = 1e3 м^2 и V = 1e-3 м^3
SA2 = 1e3; SA1 = SA2 / area_ratio; V = 0.1**3
check('стр. 227: доли A  S1:S2:объём = 2.2e-5 : 1 : 1.4e-9 (при S2=1e3 м^2, V=1e-3 м^3)',
      within(st['n1A']*SA1/(st['n2A']*SA2), 2.2e-5, 0.15) and within(st['ncA']*V/(st['n2A']*SA2), 1.4e-9, 0.15))
check('стр. 227: доли A2 S1:S2:объём = 1.3e-15 : 1 : 6.3e-6',
      within(st['n1A2']*SA1/(st['n2A2']*SA2), 1.3e-15, 0.15) and within(ncA2_ref*V/(st['n2A2']*SA2), 6.3e-6, 0.15))
check('стр. 226: загрузки ~4e15 атомов A и 3.2e18 молекул A2',
      within(st['n2A']*SA2, 4e15, 0.15) and within(st['n2A2']*SA2, 3.2e18, 0.15))
lam = 1 / (pi * rA**2 * 2e17)
print('Limit 2 (для информации): lambda = 1/(pi r_A^2 n) при 2e17 м^-3 = %.2g м; в книге «~1 м» (=10 L_cav)' % lam)

[OK] баланс атомов на S1: два выражения J совпадают   (3.95e+18 / 3.95e+18)
[OK] (7.26) = (7.25) точно при m_A2 v_A2 = sqrt2 m_A v_A   (8.45295e-05 vs 8.45295e-05 Па)
[OK] баланс потоков: R_des(i,A) + 2 R_des(i,A2) = R_T на S1 и S2
  с табличными скоростями (m_A2 v_A2 / m_A v_A = 1.4364 вместо sqrt2): (7.25) = 8.133e-05 Па, т.е. на 3.8% меньше (7.26)
[OK] dP совпадает с прежним 8.45e-5 Па (1%)   (8.453e-05)
[OK] alpha1, alpha2 совпадают с прежними (2%)   (1.49e-13, 2.73e+06)
[OK] циклический ток J совпадает с прежним 3.95e18 (1%)   (3.95e+18 м^-2 с^-1)
тепло, поглощаемое S1 (и выделяемое S2) при одной T: 1.57 Вт/м^2; на пятно 1e-6 м^2: 1.57e-06 Вт
dP/P = 0.306
[OK] стр. 225: dP(2e14) ~ 8e-7 Па и dP(2e17) ~ 8e-4 Па (в пределах x1.5)   (8.45e-07; 0.000845 Па)
[OK] стр. 227: доли A  S1:S2:объём = 2.2e-5 : 1 : 1.4e-9 (при S2=1e3 м^2, V=1e-3 м^3)
[OK] стр. 227: доли A2 S1:S2:объём = 1.3e-15 : 1 : 6.3e-6
[OK] стр. 226: загрузки ~4e15 атомов A и 3.2e18 молекул A2
Limit 2 (для информации): lam

## 4. При $W=1$ эффект исчезает: $\Delta P\to0$, $J\to0$, $\alpha(1)=\alpha(2)$

Чтобы получить $W=1$, правлю **одну** предэкспоненту и оставляю все энергии без изменений. Три независимых варианта:
- **A:** $\gamma_{diss}(1)\to\gamma_{diss}(1)/W$ (= $10^{-27}$, допустимая вероятность), т.е. $K(1)\to K(1)\,W$;
- **B:** $\gamma_{recomb}(2)\to\gamma_{recomb}(2)/W$, т.е. $K(2)\to K(2)/W$ (меняется и состав газа);
- **C:** $F(2,A)\to F(2,A)/\sqrt{W}=10^{-10}$, т.е. $H_A(2)^2\to H_A(2)^2/W$. B и C дают один и тот же газ (у S2 при этом разные поверхностные инвентари), поэтому числа у них совпадают.

Ожидание: S1 находится в равновесии с газом, заданным S2, т.е. $n(1,A)=H_A(1)\,n(c,A)$ и $\alpha(1)=\alpha(2)=\Gamma_{A_2}/\Gamma_A$. Затем — скан по $W$: знак ΔP должен совпадать со знаком $\ln W$, а вблизи $W=1$ должен быть линейный отклик $\Delta P\propto\ln W$, т.е. $\ln W$ играет роль термодинамической силы цикла.

In [10]:
def variant(name):
    p1, p2 = dict(S[1]), dict(S[2])
    if name == 'A': p1['gd'] = S[1]['gd'] / W
    if name == 'B': p2['gr'] = S[2]['gr'] / W
    if name == 'C': p2['FA'] = S[2]['FA'] / sqrt(W)
    return surface(p1), surface(p2)

for name in ('A', 'B', 'C'):
    s1, s2 = variant(name)
    Wv = s2['Kg'] / s1['Kg']
    r = steady(ncA2_ref, s1, s2)
    agas = r['GA2'] / r['GA']
    print('вариант %s: W=%.15g  a1=%.6g  a2=%.6g  dP=%.3g Па  J=%.3g' % (name, Wv, r['a1'], r['a2'], r['dP25'], r['J']))
    check('%s: W = 1' % name, abs(Wv - 1) < 1e-9)
    check('%s: S1 в равновесии с газом: n(1,A) = H_A(1) n(c,A)' % name, rel(r['n1A'], s1['HA'] * r['ncA']) < 1e-9)
    check('%s: alpha(1) = alpha(2) = Gamma_A2/Gamma_A' % name, rel(r['a1'], r['a2']) < 1e-9 and rel(r['a2'], agas) < 1e-9)
    check('%s: |dP|/P < 1e-12 (по 7.25 и 7.26)' % name, abs(r['dP25']) / r['P'] < 1e-12 and abs(r['dP26']) / r['P'] < 1e-12,
          '%.2g; %.2g' % (r['dP25'] / r['P'], r['dP26'] / r['P']))
    check('%s: |J|/Gamma_A2 < 1e-9' % name, abs(r['J']) / r['GA2'] < 1e-9, '%.2g' % (r['J'] / r['GA2']))

вариант A: W=1  a1=2.73221e+06  a2=2.73221e+06  dP=-2.71e-20 Па  J=512
[OK] A: W = 1
[OK] A: S1 в равновесии с газом: n(1,A) = H_A(1) n(c,A)
[OK] A: alpha(1) = alpha(2) = Gamma_A2/Gamma_A
[OK] A: |dP|/P < 1e-12 (по 7.25 и 7.26)   (-9.8e-17; 9.5e-24)
[OK] A: |J|/Gamma_A2 < 1e-9   (1.3e-16)
вариант B: W=1  a1=2.73221e-07  a2=2.73221e-07  dP=-1.14e-13 Па  J=2.05e+03
[OK] B: W = 1
[OK] B: S1 в равновесии с газом: n(1,A) = H_A(1) n(c,A)
[OK] B: alpha(1) = alpha(2) = Gamma_A2/Gamma_A
[OK] B: |dP|/P < 1e-12 (по 7.25 и 7.26)   (-1.6e-16; 2.3e-23)
[OK] B: |J|/Gamma_A2 < 1e-9   (5.2e-16)
вариант C: W=1  a1=2.73221e-07  a2=2.73221e-07  dP=-1.14e-13 Па  J=2.05e+03
[OK] C: W = 1
[OK] C: S1 в равновесии с газом: n(1,A) = H_A(1) n(c,A)
[OK] C: alpha(1) = alpha(2) = Gamma_A2/Gamma_A
[OK] C: |dP|/P < 1e-12 (по 7.25 и 7.26)   (-1.6e-16; 2.3e-23)
[OK] C: |J|/Gamma_A2 < 1e-9   (5.2e-16)


In [11]:
def scan(logW):
    p1 = dict(S[1]); p1['gd'] = S[1]['gd'] * 10**logW / W     # даёт W' = 10**logW
    s1 = surface(p1)
    return sf[2]['Kg'] / s1['Kg'], steady(ncA2_ref, s1, sf[2])

print(' log10 W     alpha1         dP, Па        J, м^-2 с^-1')
rows = []
for lw in (-6, -3, -1, -0.1, 0, 0.1, 1, 3, 6, 13, 20, 26):
    Wp, r = scan(lw)
    rows.append((lw, Wp, r))
    print('%8.2f  %12.4g  %13.4g  %13.4g' % (lw, r['a1'], r['dP25'], r['J']))
check("знак dP = знак ln W' (|log W'| >= 0.1)", all((r['dP25'] > 0) == (lw > 0) for lw, Wp, r in rows if abs(lw) >= 0.1))
check("знак J = знак ln W'", all((r['J'] > 0) == (lw > 0) for lw, Wp, r in rows if abs(lw) >= 0.1))
check("dP монотонно растёт с ln W'", all(rows[k][2]['dP25'] < rows[k+1][2]['dP25'] for k in range(len(rows) - 1)))
print("  замечание: при log10 W' > ~15 alpha(1) -> 0 и dP насыщается на 8.13e-5 Па — книжная точка W = 1e26 лежит в насыщении")
check("dP = 0 при W' = 1", abs([r for lw, Wp, r in rows if lw == 0][0]['dP25']) / st['P'] < 1e-12)
eps = 1e-3
_, rp = scan(eps); _, rm = scan(-eps)
check("линейный отклик: dP(+eps) = -dP(-eps) (eps = 1e-3 по log10 W, 1%)", within(-rm['dP25'], rp['dP25'], 0.01),
      '%.4g vs %.4g Па' % (rp['dP25'], rm['dP25']))

 log10 W     alpha1         dP, Па        J, м^-2 с^-1
   -6.00     2.732e+09     -1.487e-11     -7.221e+11
   -3.00      8.64e+07     -1.441e-11         -7e+11
   -1.00      8.64e+06     -1.018e-11     -4.943e+11
   -0.10     3.066e+06     -1.619e-12     -7.861e+10
    0.00     2.732e+06     -2.711e-20            512
    0.10     2.435e+06      1.816e-12       8.82e+10
    1.00      8.64e+05      3.218e-11      1.563e+12
    3.00      8.64e+04      4.558e-10      2.214e+13
    6.00          2732      1.487e-08      7.221e+14
   13.00        0.6494      3.538e-05      1.718e+18
   20.00     1.493e-07      8.133e-05       3.95e+18
   26.00     1.493e-13      8.133e-05       3.95e+18
[OK] знак dP = знак ln W' (|log W'| >= 0.1)
[OK] знак J = знак ln W'
[OK] dP монотонно растёт с ln W'
  замечание: при log10 W' > ~15 alpha(1) -> 0 и dP насыщается на 8.13e-5 Па — книжная точка W = 1e26 лежит в насыщении
[OK] dP = 0 при W' = 1
[OK] линейный отклик: dP(+eps) = -dP(-eps) (eps = 1e-3 по log10 W

## 5. Ур. 7.4 ($P_{des}=P_{ads}$) против 7.20: больцмановский множитель

В §7.2 при $P_{des}=P_{ads}$ ур. 7.2–7.3 дают $H_j=v_j/(4\nu_0)$ — длину порядка $10^{-11}$ м, **не зависящую от энергии связи**. Отсюда (7.6): $K(g)=\dfrac{v_{A_2}}{2\nu_0}K(s)$, и цикл двух полостей даёт $W_{7.6}=K(2)/K(1)$.
В §7.3 те же авторы используют (7.20), где вероятность десорбции за попытку равна $e^{-\Delta E/kT}/F$, т.е. больцмановский множитель есть. Правильная форма микрообратимости: $P_{des}=P_{ads}\times(\text{отношение статсумм})\times e^{-\Delta E_{des}/kT}$. При ней все $H$ и $K$ выражаются через статсуммы, и $W\equiv1$ тождественно.

In [12]:
H74 = {'A': vA / (4 * nu0), 'A2': vA2 / (4 * nu0)}
Kg76 = {i: vA2 / (2 * nu0) * sf[i]['K'] for i in (1, 2)}
Kg74 = {i: sf[i]['K'] * H74['A']**2 / H74['A2'] for i in (1, 2)}
vAc = sqrt(2) * vA2                                   # согласованные скорости
Kg74c = {i: sf[i]['K'] * (vAc / (4 * nu0))**2 / H74['A2'] for i in (1, 2)}
check('(7.6) = K*H_A^2/H_A2 с H = v/(4 nu0) точно при v_A = sqrt2 v_A2', all(rel(Kg76[i], Kg74c[i]) < 1e-12 for i in (1, 2)))
print('  с табличными скоростями отличие %.1f%% (v_A^2/(2 v_A2^2) = %.4f)' % (100*(1 - Kg74[1]/Kg76[1]), vA**2/(2*vA2**2)))
W76 = Kg76[2] / Kg76[1]
check('уже по (7.6): W = K(2)/K(1) != 1', abs(log10(W76)) > 1, 'W_7.6 = %.3g' % W76)
for i in (1, 2):
    print('S%d: Kg по (7.6) = %.3g м^3,  Kg по §7.3 = %.3g м^3,  расхождение 10^%.1f'
          % (i, Kg76[i], sf[i]['Kg'], log10(sf[i]['Kg'] / Kg76[i])))
ratio = sf[1]['HA'] / H74['A']
check('H(7.20)/H(7.4) = F*exp(E_des/kT) (S1, A)', rel(ratio, S[1]['FA'] * exp(S[1]['EdA'] * 1e3 / (Rg * T0))) < 1e-12, '%.3g' % ratio)


[OK] (7.6) = K*H_A^2/H_A2 с H = v/(4 nu0) точно при v_A = sqrt2 v_A2
  с табличными скоростями отличие 3.1% (v_A^2/(2 v_A2^2) = 0.9694)
[OK] уже по (7.6): W = K(2)/K(1) != 1   (W_7.6 = 3.69e+14)
S1: Kg по (7.6) = 2.17e-35 м^3,  Kg по §7.3 = 7.24e-30 м^3,  расхождение 10^5.5
S2: Kg по (7.6) = 8.02e-21 м^3,  Kg по §7.3 = 0.000724 м^3,  расхождение 10^17.0
[OK] H(7.20)/H(7.4) = F*exp(E_des/kT) (S1, A)   (1.14e+11)


### 5а. Ориентир: «настоящая» газовая константа $K_{eq}$ для гипотетического $A_2$

Это оценка, а не проверка книги. Длина связи и электронные вырождения в книге не заданы, поэтому я варьирую $r_e\in[1,10]$ Å и электронный множитель $g_{A_2}/g_A^2\in[1/16,4]$; энергию 240 кДж/моль считаю $D_0$, колебания — гармоническими. Устойчивый вывод: $K_g(1)\ll K_{eq}\ll K_g(2)$ при любых значениях из этого диапазона. Обе поверхности «промахиваются» мимо газовой константы в противоположные стороны.

In [13]:
def Keq_gas(re, gel, T=T0):
    lamA = h / sqrt(2 * pi * mA * kB * T); lamA2 = h / sqrt(2 * pi * mA2 * kB * T)
    I = (mA / 2) * re**2
    qrot = 8 * pi**2 * I * kB * T / (2 * h**2)
    qvib = 1 / (1 - exp(-h * nuvib / (kB * T)))
    return gel * lamA**6 / lamA2**3 * qrot * qvib * exp(EAA * 1e3 / (Rg * T))
vals = [Keq_gas(re, g) for re in (1e-10, 2e-10, 5e-10, 1e-9) for g in (1/16., 1., 4.)]
print('K_eq в диапазоне %.2g ... %.2g м^3 (r_e = 2 Å, g = 1: %.2g)' % (min(vals), max(vals), Keq_gas(2e-10, 1.)))
check('Kg(1) < min K_eq и max K_eq < Kg(2) — обе поверхности не согласованы с газом',
      sf[1]['Kg'] < min(vals) and max(vals) < sf[2]['Kg'],
      'log10: Kg1=%.1f, K_eq=[%.1f, %.1f], Kg2=%.1f' % (log10(sf[1]['Kg']), log10(min(vals)), log10(max(vals)), log10(sf[2]['Kg'])))

K_eq в диапазоне 6.2e-19 ... 4e-15 м^3 (r_e = 2 Å, g = 1: 4e-17)
[OK] Kg(1) < min K_eq и max K_eq < Kg(2) — обе поверхности не согласованы с газом   (log10: Kg1=-29.1, K_eq=[-18.2, -14.4], Kg2=-3.1)


## 6. Проверка вычислимых утверждений из ноутбука пользователя

Файл `chemical_nonequilibrium_steady_states.ipynb` состоит из 16 ячеек: 12 markdown с текстом, 6 из них содержат только рисунки книги, и 2 пустые код-ячейки. Своих вычислений и параметров в нём нет. Ниже проверяются его утверждения, которые можно посчитать. Подробный разбор по ячейкам — в `review_user_notebook.md`.

In [14]:
import json, os
cand = ['chemical_nonequilibrium_steady_states.ipynb', 'tmp/dspg/chemical_nonequilibrium_steady_states.ipynb']
wd = globals().get('WORKSPACE_DIR')
if wd: cand.append(os.path.join(wd, 'tmp/dspg/chemical_nonequilibrium_steady_states.ipynb'))
path = next((c for c in cand if os.path.exists(c)), None)
if path is None:
    print('[SKIP] файл пользователя не найден рядом с ноутбуком')
else:
    unb = json.load(open(path))
    srcs = [(c['cell_type'], ''.join(c['source']) if isinstance(c['source'], list) else c['source']) for c in unb['cells']]
    n_code = sum(1 for t, s in srcs if t == 'code'); n_code_nonempty = sum(1 for t, s in srcs if t == 'code' and s.strip())
    alltext = '\n'.join(s for t, s in srcs)
    print('ячеек: %d, код-ячеек: %d, непустых код-ячеек: %d' % (len(srcs), n_code, n_code_nonempty))
    check('у пользователя нет вычислений (все код-ячейки пустые)', n_code_nonempty == 0)
    check('в тексте пользователя нет обсуждения exp(-E/kT) / Больцмана / Вегшайдера в (7.4)',
          not any(w in alltext for w in ('\\exp', 'e^{-', 'kT', 'Больцман', 'больцман', 'Вегшайдер')))
    check('пользователь воспроизводит (7.4) P_des = P_ads и (7.6) без оговорок',
          'P_{des}(A) = P_{ads}(A)' in alltext and 'K(g) = \\frac{v_{A_2}}{2 \\nu_0} K(s)' in alltext)

ячеек: 16, код-ячеек: 2, непустых код-ячеек: 0
[OK] у пользователя нет вычислений (все код-ячейки пустые)
[OK] в тексте пользователя нет обсуждения exp(-E/kT) / Больцмана / Вегшайдера в (7.4)
[OK] пользователь воспроизводит (7.4) P_des = P_ads и (7.6) без оговорок


**У1. Алгебра (7.5)→(7.6), которую пользователь приводит.** При $v_A=\sqrt2\,v_{A_2}$ и равных $\nu$ из (7.5) получается (7.6). В книге условие напечатано как «$v_A^2=2v_{A_2}$» — это опечатка, должно быть $v_A^2=2v_{A_2}^2$.

**У2. Утверждение пользователя «поскольку $\nu_0$ зависит от материала, $K(g)$ определяется поверхностью».** В книге сказано обратное: «$\nu_{att}\simeq\nu_o$ is relatively independent of surface type», а от поверхности зависит $K(s)$ (стр. 217). В табл. 7.1 одна $\nu_0$ на обе поверхности. Значит, по (7.6) отношение $K(g)$ двух полостей целиком задаётся $K(s)$.

In [15]:
Ks, nu, v2 = 3.7e-12, 2.3e13, 650.0
v1 = sqrt(2) * v2
lhs = 0.25 * v1**2 / v2 * nu / nu**2 * Ks          # (7.5)
rhs = v2 / (2 * nu) * Ks                            # (7.6)
check('У1: (7.5) -> (7.6) при v_A = sqrt2 v_A2 (точно)', rel(lhs, rhs) < 1e-12)
v1t = sqrt(2 * v2)                                  # буквально «v_A^2 = 2 v_A2» в СИ
lhs_t = 0.25 * v1t**2 / v2 * nu / nu**2 * Ks
check('У1: при буквальном «v_A^2 = 2 v_A2» (опечатка книги) (7.6) НЕ получается; верно v_A^2 = 2 v_A2^2',
      rel(lhs_t, rhs) > 0.5, 'lhs/rhs = %.3g' % (lhs_t / rhs))
check('У2: при общей nu0 отношение K(g) двух полостей по (7.6) = K(2)/K(1), nu0 сокращается',
      rel(Kg76[2] / Kg76[1], sf[2]['K'] / sf[1]['K']) < 1e-12, '%.3g' % (Kg76[2] / Kg76[1]))
print('  -> вывод пользователя «через nu0» книгой НЕ подтверждается: зависимость от поверхности входит через K(s)')

[OK] У1: (7.5) -> (7.6) при v_A = sqrt2 v_A2 (точно)
[OK] У1: при буквальном «v_A^2 = 2 v_A2» (опечатка книги) (7.6) НЕ получается; верно v_A^2 = 2 v_A2^2   (lhs/rhs = 0.00154)
[OK] У2: при общей nu0 отношение K(g) двух полостей по (7.6) = K(2)/K(1), nu0 сокращается   (3.69e+14)
  -> вывод пользователя «через nu0» книгой НЕ подтверждается: зависимость от поверхности входит через K(s)


**У3. Больцмановский множитель в $P_{des}/P_{ads}$ (7.4).** Пользователь переписывает (7.4) как есть. Проверка: при (7.4) $H=v/(4\nu_0)$ одно и то же для энергии связи 1 и 400 кДж/моль, т.е. покрытие не зависит от энергии связи, что абсурдно. По (7.20) отношение этих двух $H$ равно $e^{399\,\text{кДж}/RT}$.

In [16]:
H_74 = lambda E: vA / (4 * nu0)                                  # (7.4): от E не зависит
H_720 = lambda E: vA * tau_des(1.0, E) / 4                       # (7.20), F = 1
check('У3: по (7.4) H(1 кДж) = H(400 кДж)', H_74(1.) == H_74(400.))
check('У3: по (7.20) H(400)/H(1) = exp(399e3/RT) ~ 10^20.8',
      rel(H_720(400.) / H_720(1.), exp(399e3 / (Rg * T0))) < 1e-9, '10^%.1f' % log10(H_720(400.) / H_720(1.)))

[OK] У3: по (7.4) H(1 кДж) = H(400 кДж)
[OK] У3: по (7.20) H(400)/H(1) = exp(399e3/RT) ~ 10^20.8   (10^20.8)


**У4. «2200 K, 2.5 Торр, стационарная разность давлений 1 Па, это 0.002 от общего давления».**
(а) Арифметика: $1\ \text{Па}/2.5\ \text{Торр}=0.0030$, а не 0.002. Неточность есть уже в книге (стр. 232), пользователь её перенёс.
(б) Разность давлений **не измерялась** («direct measurements of pressure were not made», стр. 234). Её **вычислили** по (7.27) из разности мощностей диссоциации. Пересчитываю (7.27) по моим считываниям рис. 7.8 (точность ±3 Вт): одиночный нагрев Mo ≈ 168 Вт, W ≈ 143 Вт; при d = 33 мм и совместном нагреве 154 и 126 Вт. Площадь ленты 6 см × 3.3 мм с одной стороны, $v_H$ — средняя скорость при 2200 K.
(в) Замечание: по той же импульсной логике, что в (7.25)–(7.26), одна диссоциация меняет поток импульса на $(2-\sqrt2)m_Hv_H$, а не на $m_Hv_H$. Значит, (7.27) завышает примерно в 1.7 раза.

In [17]:
r1 = 1.0 / (2.5 * Torr)
check('У4а: 1 Па / 2.5 Торр = 0.0030 (в книге и у пользователя 0.002)', abs(r1 - 0.0030) < 1e-4, '%.5f' % r1)
mH = 1.00794 * amu; Tf = 2200.0
vH = sqrt(8 * kB * Tf / (pi * mH)); Eb = 431e3 / NA
Afil = 0.06 * 3.3e-3
check('У4б: E_b = 431 кДж/моль = 4.47 эВ (обе цифры книги согласованы)', abs(Eb / 1.602176634e-19 - 4.47) < 0.01)
for lab, dPhd in (('одиночный нагрев, 168-143', 25.), ('d=33 мм, совместно, 154-126', 28.)):
    one = dPhd * mH * vH / (Eb * Afil); two = one / 2
    print('  %s: dF_p = %.2f Па (одна сторона), %.2f Па (две стороны); с (2-sqrt2): %.2f / %.2f Па'
          % (lab, one, two, (2 - sqrt(2)) * one, (2 - sqrt(2)) * two))
dF = 25. * mH * vH / (Eb * Afil)
check('У4б: (7.27) даёт величину ~1 Па (0.5...2.5 Па) — число книги воспроизводится как ВЫЧИСЛЕННОЕ', 0.5 < dF / 2 < 2.5 and 0.5 < dF < 2.5,
      'v_H = %.0f м/с' % vH)

[OK] У4а: 1 Па / 2.5 Торр = 0.0030 (в книге и у пользователя 0.002)   (0.00300)
[OK] У4б: E_b = 431 кДж/моль = 4.47 эВ (обе цифры книги согласованы)
  одиночный нагрев, 168-143: dF_p = 2.01 Па (одна сторона), 1.00 Па (две стороны); с (2-sqrt2): 1.18 / 0.59 Па
  d=33 мм, совместно, 154-126: dF_p = 2.25 Па (одна сторона), 1.12 Па (две стороны); с (2-sqrt2): 1.32 / 0.66 Па
[OK] У4б: (7.27) даёт величину ~1 Па (0.5...2.5 Па) — число книги воспроизводится как ВЫЧИСЛЕННОЕ   (v_H = 6798 м/с)


**У5. «При увеличении давления водорода стационарная разность давлений сходит на ноль» (рис. 7.7).**
На рис. 7.7 отложена **разность мощностей диссоциации на 1 Торр**, $\Delta P_{hd}/P(H_2)$ [Вт/Торр], а не разность давлений. Мои считывания (±2 Вт/Торр): 0.5 → 158, 1 → 56, 2 → 22, 4 → 8, 8 → 3 Вт/Торр; при ≥16 Торр значения в масштабе графика неотличимы от нуля. По (7.27) $\Delta F_p\propto\Delta P_{hd}$, а $\Delta P_{hd}=(\Delta P_{hd}/P)\cdot P$ спадает гораздо медленнее.

In [18]:
fig77 = [(0.5, 158.), (1., 56.), (2., 22.), (4., 8.), (8., 3.)]
abs_hd = [(p, y * p) for p, y in fig77]
print('  P, Торр | dP_hd/P, Вт/Торр | dP_hd, Вт | dF_p/P(H2) ~ отн.')
for (p, y), (_, a) in zip(fig77, abs_hd):
    print('  %6.1f | %8.0f | %6.0f | %.3g' % (p, y, a, a / p / (abs_hd[0][1] / abs_hd[0][0])))
check('У5: dP_hd/P падает ~50x от 0.5 до 8 Торр', 30 < fig77[0][1] / fig77[-1][1] < 80, '%.0f x' % (fig77[0][1] / fig77[-1][1]))
check('У5: абсолютная dP_hd (~ dF_p) падает лишь ~3x на том же интервале', 2 < abs_hd[0][1] / abs_hd[-1][1] < 5,
      '%.1f x' % (abs_hd[0][1] / abs_hd[-1][1]))
print('  -> «сходит на ноль» верно для относительной величины (на Торр); для абсолютной dF_p по графику так сказать нельзя')
# согласованность рисунков при ~2200 K, 2.5 Торр, одиночный нагрев
y25 = 10**(log10(22.) + (log10(8.) - log10(22.)) * (log10(2.5) - log10(2.)) / (log10(4.) - log10(2.)))
print('  рис. 7.7 -> dP_hd(2.5 Торр) ~ %.0f Вт; рис. 7.8 (пунктир) -> 25 Вт; рис. 7.6 (2125/2250 K) -> 40/53 Вт' % (y25 * 2.5))
check('У5: рисунки 7.6–7.8 согласованы по dP_hd лишь в пределах ~2x (по моим считываниям)', 1.3 < y25 * 2.5 / 25. < 2.5,
      'отношение %.2f' % (y25 * 2.5 / 25.))

  P, Торр | dP_hd/P, Вт/Торр | dP_hd, Вт | dF_p/P(H2) ~ отн.
     0.5 |      158 |     79 | 1
     1.0 |       56 |     56 | 0.354
     2.0 |       22 |     44 | 0.139
     4.0 |        8 |     32 | 0.0506
     8.0 |        3 |     24 | 0.019
[OK] У5: dP_hd/P падает ~50x от 0.5 до 8 Торр   (53 x)
[OK] У5: абсолютная dP_hd (~ dF_p) падает лишь ~3x на том же интервале   (3.3 x)
  -> «сходит на ноль» верно для относительной величины (на Торр); для абсолютной dF_p по графику так сказать нельзя
  рис. 7.7 -> dP_hd(2.5 Торр) ~ 40 Вт; рис. 7.8 (пунктир) -> 25 Вт; рис. 7.6 (2125/2250 K) -> 40/53 Вт
[OK] У5: рисунки 7.6–7.8 согласованы по dP_hd лишь в пределах ~2x (по моим считываниям)   (отношение 1.59)


**У6. «При температурах окружающей среды выход атомарного водорода в газ пренебрежимо мал»** (ячейка 14).
Проверяю по данным NIST-JANAF для H(g): $\Delta_fG^\circ(298.15\,\text{K})=203.278$, $\Delta_fG^\circ(300)=203.186$, $\Delta_fG^\circ(2200)=94.712$ кДж/моль. Таблица `janaf.nist.gov/tables/H-001.txt` получена в этой сессии. $K_p(H_2\rightleftharpoons2H)=\exp(-2\Delta_fG^\circ_H/RT)$, $p^\circ=1$ бар.

In [19]:
def xH(dGf_kJ, T, p_Pa):
    Kp = exp(-2 * dGf_kJ * 1e3 / (Rg * T)); pr = p_Pa / 1e5
    # pH^2 = Kp*(pr - pH)
    pH = (-Kp + sqrt(Kp * Kp + 4 * Kp * pr)) / 2
    return pH / pr
x300 = xH(203.186, 300., 1e5); x2200 = xH(94.712, 2200., 2.5 * Torr)
print('  x_H(300 K, 1 бар) = 10^%.1f;  x_H(2200 K, 2.5 Торр) = %.3f' % (log10(x300), x2200))
check('У6: при 300 K равновесная доля H < 1e-30 — утверждение пользователя ВЕРНО', x300 < 1e-30)
check('У6: при 2200 K, 2.5 Торр равновесная доля H ~ 10% (то есть в опыте Sheehan диссоциация не мала)', 0.05 < x2200 < 0.2)

  x_H(300 K, 1 бар) = 10^-35.4;  x_H(2200 K, 2.5 Торр) = 0.093
[OK] У6: при 300 K равновесная доля H < 1e-30 — утверждение пользователя ВЕРНО
[OK] У6: при 2200 K, 2.5 Торр равновесная доля H ~ 10% (то есть в опыте Sheehan диссоциация не мала)


**У7. Выполняется ли Вегшайдер «в параметрах пользователя».** Своих параметров у пользователя нет. Его единственная формула (7.6), применённая к двум полостям, даёт $W=K(2)/K(1)$, т.е. нарушение при любых $K(1)\ne K(2)$; с табл. 7.2 $W_{7.6}=3.7\cdot10^{14}$. Модель скоростей §7.3 с таблицами книги даёт $W=10^{26}$ (п. 2).

**У8. План пользователя (ячейка 14): ВД2 на «отклонениях от детального равновесия при сорбции/десорбции H₂» в сверхпроницаемой мембране.** Это проверяется на минимальной модели. Мембрана с гранями a и b; с обеих сторон H₂ при одной $T$ и одном $p$. Поглощение на грани идёт с вероятностью $s_x$, рекомбинационная десорбция — с константой $k_x$, в объёме — диффузия. Равновесная растворимость у грани $c_{eq,x}^2=s_x\Gamma/k_x$. Для цикла $H_2(a)\to2H(\text{объём})\to H_2(b)$ получается $W=(s_a/k_a)/(s_b/k_b)$. Асимметрия кинетики ($s_a\ne s_b$) при $W=1$ потока не даёт. Поток при $p_a=p_b$ возникает **только** при $W\ne1$, и его знак совпадает со знаком $\ln W$.

In [20]:
check('У7: W по (7.6) для табл. 7.2 = K(2)/K(1) ~ 3.7e14', within(W76, 3.7e14, 0.02), '%.3g' % W76)

def membrane_J(sa, ka, sb, kb, Ga, Gb, DL):
    # J (атомы/м^2/с, a->b): J = 2 sa Ga - 2 ka ca^2 = DL (ca - cb) = 2 kb cb^2 - 2 sb Gb
    g = lambda J: DL * (sqrt((2*sa*Ga - J) / (2*ka)) - sqrt((2*sb*Gb + J) / (2*kb))) - J
    lo, hi = -2 * sb * Gb, 2 * sa * Ga
    for _ in range(300):
        mid = 0.5 * (lo + hi)
        if g(mid) > 0: lo = mid
        else: hi = mid
    return 0.5 * (lo + hi)

G = 1e20; sa, ka = 1e-1, 1e-25; sb = 1e-4; DL = 1e-3
J_W1 = membrane_J(sa, ka, sb, ka * sb / sa, G, G, DL)              # W = 1, но s_a/s_b = 1000
J_dp = membrane_J(sa, ka, sb, ka * sb / sa, 2 * G, G, DL)          # W = 1, p_a = 2 p_b
check('У8: W=1, асимметричная кинетика (s_a/s_b=1e3), p_a=p_b  ->  J = 0', abs(J_W1) / (2 * sb * G) < 1e-9, 'J = %.2g' % J_W1)
check('У8: W=1, p_a = 2 p_b  ->  J > 0 (обычная проницаемость по Δμ)', J_dp > 0, 'J = %.3g' % J_dp)
for Wm in (10., 0.1):
    Jm = membrane_J(sa, ka, sb, ka * sb / sa * Wm, G, G, DL)     # W = (s_a/k_a)/(s_b/k_b) = Wm
    check('У8: W=%g, p_a=p_b  ->  J != 0, знак = знак ln W' % Wm, (Jm > 0) == (Wm > 1) and abs(Jm) > 1e-6 * sb * G, 'J = %.3g' % Jm)
print('  -> для одного материала мембраны и гиббсовой стенки s/k на обеих гранях связано одной константой растворимости,')
print('     т.е. W = 1 и тока нет при любых s_a != s_b. Ток требует W != 1 — ровно того же, что в DSPG.')

[OK] У7: W по (7.6) для табл. 7.2 = K(2)/K(1) ~ 3.7e14   (3.69e+14)
[OK] У8: W=1, асимметричная кинетика (s_a/s_b=1e3), p_a=p_b  ->  J = 0   (J = 4.9e-72)
[OK] У8: W=1, p_a = 2 p_b  ->  J > 0 (обычная проницаемость по Δμ)   (J = 1.99e+16)
[OK] У8: W=10, p_a=p_b  ->  J != 0, знак = знак ln W   (J = 1.72e+17)
[OK] У8: W=0.1, p_a=p_b  ->  J != 0, знак = знак ln W   (J = -1.8e+16)
  -> для одного материала мембраны и гиббсовой стенки s/k на обеих гранях связано одной константой растворимости,
     т.е. W = 1 и тока нет при любых s_a != s_b. Ток требует W != 1 — ровно того же, что в DSPG.


## 7. Итог

In [21]:
n_ok = sum(1 for n, s in CHECKS if s == 'OK'); n_fail = sum(1 for n, s in CHECKS if s == 'FAIL')
print('ИТОГ: %d проверок, OK = %d, FAIL = %d' % (len(CHECKS), n_ok, n_fail))
for n, s in CHECKS:
    if s == 'FAIL': print('  FAIL:', n)
print('W = Kg(2)/Kg(1) = %.6g (точно 10^26, не зависит от T); dP = %.3g Па; J = %.3g м^-2 с^-1' % (W, st['dP26'], st['J']))
print('при W = 1 (три варианта правки предэкспонент): dP -> 0, J -> 0, alpha(1) = alpha(2)')

ИТОГ: 92 проверок, OK = 92, FAIL = 0
W = Kg(2)/Kg(1) = 1e+26 (точно 10^26, не зависит от T); dP = 8.45e-05 Па; J = 3.95e+18 м^-2 с^-1
при W = 1 (три варианта правки предэкспонент): dP -> 0, J -> 0, alpha(1) = alpha(2)


**Выводы.**
1. Условие Вегшайдера для параметров табл. 7.1–7.2 нарушено: $W=K_g(2)/K_g(1)=10^{26}$ ровно. Энергии согласованы по Гессу, а нарушение целиком сидит в предэкспонентах $F$ и $\gamma$. Число не зависит от $T$, скоростей и частот. Это совпадает с прежней фоновой задачей; расхождений нет, кроме долей процента из-за констант.
2. Табл. 7.3, отношения потоков (стр. 227), инвентари (стр. 226–227; они неявно задают $S_2=10^3$ м², $V=10^{-3}$ м³) и диапазон ΔP (стр. 225) воспроизводятся из (7.15)–(7.21) в пределах 15%. Формула (7.26) выводится из (7.25).
3. Если одной предэкспонентой выставить $W=1$ (любой из трёх вариантов), то $\alpha(1)=\alpha(2)$, $\Delta P=0$ и $J=0$ с машинной точностью. Знаки ΔP и $J$ совпадают со знаком $\ln W$, а вблизи $W=1$ отклик линеен. Таким образом, DSPG — это ровно ток, вызванный «силой» $\ln W\ne0$, заложенной в параметры.
4. Ур. 7.4 теряет больцмановский множитель и отношение статсумм. Уже (7.6) даёт $W=K(2)/K(1)\ne1$, а с (7.20) расхождение доходит до 17 порядков.
5. Пользователь пересказал §7.1–7.2 и §7.5 без вычислений. Ошибки — в трактовке $\nu_0$, слове «наблюдалось» для ΔP и чтении рис. 7.7. Утверждение о малой равновесной доле H при комнатной $T$ верно. Минимальная модель мембраны показывает, что его план даёт ток только при $W\ne1$.